# TP - IRVE

## Partie 1 - Profiling & audit

Travail realise par :
- Issam Belhorma
- Zakaria Soual

### Ce que l'on fait dans cette partie

Dans cette premiere partie, on repond simplement aux demandes du support :
- charger le CSV consolide depuis `data.gouv.fr` ;
- generer un rapport `ydata-profiling` dans le notebook ;
- remplir une petite grille d'audit avec les problemes principaux ;
- calculer un score de qualite global initial.

On ne nettoie pas encore le dataset ici : on mesure seulement son etat de depart.

In [ ]:
%pip install -r requirements.txt

### Chargement des donnees

On charge directement l'URL officielle, parce que le sujet demande explicitement de travailler sur le fichier consolide publie sur `data.gouv.fr`.

In [ ]:
import numpy as np
import pandas as pd
from ydata_profiling import ProfileReport

# URL officielle donnee dans l'enonce.
CSV_URL = 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'

# low_memory=False permet d'avoir des types plus stables, ce qui est preferable pour un audit.
df = pd.read_csv(CSV_URL, sep=',', low_memory=False)

# On verifie rapidement la taille du fichier pour confirmer qu'on travaille bien sur la bonne livraison.
print(f"Nombre de lignes : {df.shape[0]:,}".replace(',', ' '))
print(f'Nombre de colonnes : {df.shape[1]}')

df.head()

### Rapport de profiling automatique

Le profiling automatique sert de premier diagnostic : il donne une vue d'ensemble sur les 52 colonnes avant l'audit manuel.

In [ ]:
# On garde le rapport dans le notebook, sans export HTML.
profile = ProfileReport(
    df,
    title='Partie 1 - Profiling initial IRVE',
    explorative=True,
    minimal=True,
)

profile.to_widgets()

### Audit simple des problemes principaux

Ici, on reste volontairement simple.

On commence par regarder la completude de toutes les colonnes, puis on construit une petite grille d'audit sur quelques problemes tres visibles du fichier :
- codes postaux manquants ;
- dates de mise en service manquantes ;
- puissances aberrantes ;
- identifiants repetes ;
- coordonnees signalees comme douteuses ;
- dates tres anciennes.

In [ ]:
# Cette table donne une vue simple de la completude sur les 52 colonnes.
# C'est utile parce qu'un audit academique commence souvent par les valeurs manquantes.
completude = pd.DataFrame({
    'colonne': df.columns,
    'nb_manquants': df.isna().sum().values,
    'pct_manquants': (df.isna().mean() * 100).round(2).values,
})

completude = completude.sort_values('pct_manquants', ascending=False).reset_index(drop=True)
completude

In [ ]:
# On convertit d'abord quelques colonnes utiles pour les controles.
# Cela permet d'ecrire des regles simples et lisibles.
date_maj = pd.to_datetime(df['date_maj'], errors='coerce')
date_mise_en_service = pd.to_datetime(df['date_mise_en_service'], errors='coerce')
puissance_nominale = pd.to_numeric(df['puissance_nominale'], errors='coerce')


# Petite fonction de gravite simple pour rendre la grille plus lisible.
def gravite(pct):
    if pct >= 40:
        return 'Critique'
    if pct >= 20:
        return 'Elevee'
    if pct >= 5:
        return 'Moyenne'
    return 'Faible'


# Cette grille est volontairement courte : le but est de repondre simplement aux problematiques du point 1.
grille_audit = pd.DataFrame([
    {
        'colonne': 'consolidated_code_postal',
        'dimension': 'Completude',
        'probleme': 'Code postal manquant',
        'pct_problemes': round(df['consolidated_code_postal'].isna().mean() * 100, 2),
    },
    {
        'colonne': 'date_mise_en_service',
        'dimension': 'Completude',
        'probleme': 'Date de mise en service manquante',
        'pct_problemes': round(df['date_mise_en_service'].isna().mean() * 100, 2),
    },
    {
        'colonne': 'puissance_nominale',
        'dimension': 'Validite',
        'probleme': 'Puissance <= 0 ou > 1000 kW',
        'pct_problemes': round(((puissance_nominale <= 0) | (puissance_nominale > 1000)).mean() * 100, 2),
    },
    {
        'colonne': 'id_pdc_itinerance',
        'dimension': 'Unicite',
        'probleme': 'Identifiant repete',
        'pct_problemes': round((df['id_pdc_itinerance'].duplicated(keep=False)).mean() * 100, 2),
    },
    {
        'colonne': 'consolidated_is_lon_lat_correct',
        'dimension': 'Exactitude',
        'probleme': 'Coordonnees signalees comme douteuses',
        'pct_problemes': round((df['consolidated_is_lon_lat_correct'] == False).mean() * 100, 2),
    },
    {
        'colonne': 'date_mise_en_service',
        'dimension': 'Coherence',
        'probleme': 'date_mise_en_service > date_maj',
        'pct_problemes': round((
            (date_mise_en_service.notna())
            & (date_maj.notna())
            & (date_mise_en_service > date_maj)
        ).mean() * 100, 2),
    },
    {
        'colonne': 'date_maj',
        'dimension': 'Fraicheur',
        'probleme': 'Date de mise a jour a plus de 365 jours du maximum du fichier',
        'pct_problemes': round((date_maj < (date_maj.max() - pd.Timedelta(days=365))).mean() * 100, 2),
    },
])

grille_audit['gravite'] = grille_audit['pct_problemes'].apply(gravite)
grille_audit = grille_audit.sort_values('pct_problemes', ascending=False).reset_index(drop=True)

grille_audit

In [ ]:
# On calcule un score simple par dimension.
# Idee retenue : 100 - pourcentage moyen de problemes sur la dimension.
def score_dimension(series):
    if len(series) == 0:
        return 100.0
    return round(100 - series.mean(), 2)


scores = pd.DataFrame([
    {'dimension': 'Completude', 'score_initial': score_dimension(completude['pct_manquants'])},
    {'dimension': 'Validite', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Validite', 'pct_problemes'])},
    {'dimension': 'Coherence', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Coherence', 'pct_problemes'])},
    {'dimension': 'Unicite', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Unicite', 'pct_problemes'])},
    {'dimension': 'Exactitude', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Exactitude', 'pct_problemes'])},
    {'dimension': 'Fraicheur', 'score_initial': score_dimension(grille_audit.loc[grille_audit['dimension'] == 'Fraicheur', 'pct_problemes'])},
])

score_global_initial = round(scores['score_initial'].mean(), 2)

display(scores)
print(f'Score de qualite global initial : {score_global_initial}/100')

### Fin de la partie 1

Le notebook peut ensuite continuer avec les parties suivantes du devoir. Ici, on s'est limite volontairement au diagnostic initial.